In [1]:
import sys
from pathlib import Path
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import datetime

In [2]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [3]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 266


In [4]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,3225040353072462344,24/7 Wall St.,"$25,000 in XRP vs $25,000 in S&P 500: Backtest...",https://finance.yahoo.com/markets/crypto/artic...,,"Quick Read\nA $25,000 investment in XRP at the...",Sam Daodu,6 min read,2026-05-22 16:02:26,2026-05-23 03:23:41.886219
1,4446138805452953510,24/7 Wall St.,"$5,000 in XRP at $1.37 vs Bitcoin at $77,000: ...",https://finance.yahoo.com/markets/crypto/artic...,,"Quick Read\nInvesting $5,000 at $1.37 buys 3,6...",Sam Daodu,8 min read,2026-05-22 08:36:29,2026-05-23 03:23:42.148369
2,1910953675637808579,BeInCrypto,$725 Million in Ethereum (ETH) Just Left Whale...,https://finance.yahoo.com/markets/crypto/artic...,,"Ethereum (ETH) price trades at $2,132 on May 2...",Ananda Banerjee,3 min read,2026-05-22 07:00:35,2026-05-23 03:23:42.185079
3,3378378421304511398,BeInCrypto,1 Quadrillion MAPO Minted: Bridge Exploit Cras...,https://finance.yahoo.com/markets/crypto/artic...,,MAP Protocol’s Butter Bridge suffered a severe...,Lockridge Okoth,2 min read,2026-05-20 17:03:35,2026-05-23 03:23:43.236435
4,2570045032087381483,BeInCrypto,10 Surprising Facts About Elon Musk’s $1 Trill...,https://finance.yahoo.com/markets/stocks/artic...,,Elon Musk's SpaceX filed for an initial public...,Mohammad Shahid,3 min read,2026-05-21 20:59:49,2026-05-23 03:23:42.431488


In [5]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_ingested_today(df)

    # Extract date strings (DB/delete API expects original string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [6]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(self, bucket_name, prefix_path, year, month, day, hour, minute):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(bucket_name, prefix_path, year=None, month=None, day=None, hour=None, minute=None)
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Rows scraped/inserted today (created_at), not article publish time (datetime)
            return filter_financial_news_ingested_today(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [7]:
etl = DataETL(df)
# Filter by ingestion date (created_at), not article publish time (datetime)
filtered_df = filter_financial_news_ingested_today(df)
print(
    f"Ingested today (created_at): {len(filtered_df)} | "
    f"Article published today (datetime): {len(filter_financial_news_by_date(df, date_column='datetime'))}"
)
filtered_df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at


In [8]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 0


In [9]:
export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"
if export_rows_to_s3 == True and not filtered_df.empty:
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

In [10]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [11]:
ingest_data = False
get_full_file = True
get_by_datetime = False
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2024'
    month = '08'
    day = '01'
    hour = ''
    minute = ''
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [12]:
if df_from_file is not None:
    print(df_from_file.count())

In [13]:
# Assuming df is your DataFrame
targetId = ''
if len(targetId) > 0:
    filtered_content = filtered_df.loc[filtered_df['id'] == targetId, 'content']

    # If you want to display the full content, convert it to a list or display all rows
    full_content = filtered_content.tolist()

    print(full_content)